# 03 — Erlang e dimensionamento

Do volume de chamadas previsto até **quantos agentes cada intervalo precisa**. Você vai ver a conta de uma carga, a tabela do Erlang C, e o plano completo para o próximo dia, com shrinkage.

Leia antes o [guia de dimensionamento](../docs/07-guia-de-dimensionamento.md). Execute as células de cima para baixo com `Shift + Enter`.

## 1. Dados e previsão

Carregamos as chamadas por 30 minutos e pedimos ao TimesFM os próximos dois dias (96 intervalos: domingo, com a central fechada, e segunda-feira), com a faixa de 10% a 90%.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from callcenter_lab.loading import detect_columns, prepare_series, resample_series

csv_path = Path('..') / 'data' / 'samples' / 'chamadas_sinteticas.csv'
df = pd.read_csv(csv_path)
date_col, value_col = detect_columns(df)
series = resample_series(prepare_series(df, date_col, value_col), '30min', how='sum', fill='zero')
series.tail()

In [ ]:
from callcenter_lab.model import forecast, load_forecaster

model = load_forecaster()
horizon = 96
result = forecast(model, series.to_numpy(), horizon)
future = pd.date_range(series.index[-1], periods=horizon + 1, freq='30min')[1:]
point, lower, upper = (pd.Series(v, index=future) for v in (result.point, result.lower, result.upper))
point.head()

## 2. A configuração

Tudo que o dimensionamento precisa saber, num lugar só. Os valores abaixo são **exemplos**: troque pelos da sua operação.

In [ ]:
from callcenter_lab.wfm import ShrinkageConfig, StaffingConfig

config = StaffingConfig(
    interval_minutes=30,          # intervalo da série
    service_level_pct=80,         # meta: 80% das chamadas...
    service_level_seconds=20,     # ...atendidas em até 20 s
    max_occupancy_pct=85,         # teto de ocupação dos agentes
    aht_seconds=300,              # TMA: 5 minutos
    shrinkage=ShrinkageConfig(pauses_pct=10, training_pct=3, absenteeism_pct=5, vacation_pct=5, other_pct=2),
)
config.shrinkage.total_pct

## 3. A carga de um intervalo

60 chamadas em 30 minutos, com TMA de 300 s: quantos Erlangs?

In [ ]:
from callcenter_lab.wfm import workload_erlangs

load = workload_erlangs(60, 300, 30)
print(f'{load:.1f} Erlangs')

Se as chamadas chegassem perfeitamente espaçadas, 10 agentes ficariam sempre ocupados. Como elas chegam ao acaso, 10 agentes não bastam. Veja o que o Erlang C diz para cada número de agentes:

In [ ]:
from callcenter_lab.wfm import asa, erlang_c, occupancy, service_level

rows = []
for agents in range(11, 17):
    rows.append({
        'agentes': agents,
        'chamadas que esperam (%)': round(100 * erlang_c(agents, load)),
        'nível de serviço 20 s (%)': round(100 * service_level(agents, load, 300, 20), 1),
        'espera média (s)': round(asa(agents, load, 300), 1),
        'ocupação (%)': round(100 * occupancy(agents, load)),
    })
pd.DataFrame(rows).set_index('agentes')

A meta é 80% em 20 segundos: o menor número de agentes que a cumpre é o que o dimensionamento escolhe.

In [ ]:
from callcenter_lab.wfm import required_agents

required_agents(load, 300, service_level_pct=80, service_level_seconds=20, max_occupancy_pct=85)

## 4. O plano dos próximos dias

`build_plan` faz a cadeia inteira: carga, agentes, nível de serviço esperado, ocupação e **pessoas escaladas** (já com o shrinkage). Abaixo, a segunda-feira das 8h às 14h (o domingo tem zero chamadas e zero agentes).

In [ ]:
from callcenter_lab.wfm import build_plan

plan = build_plan(point, config)
plan[['calls', 'workload_erlangs', 'agents', 'service_level_pct', 'occupancy_pct', 'scheduled']].round(1).iloc[48 + 16:48 + 28]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(plan.index, plan['agents'], width=0.015, label='Agentes atendendo')
ax.plot(plan.index, plan['scheduled'], color='tab:orange', label='Pessoas escaladas (com shrinkage)')
ax.set_ylabel('Pessoas')
ax.legend()
plt.show()

## 5. E se o volume vier diferente da previsão?

A previsão tem uma faixa de incerteza. Quantos agentes o cenário baixo (10%) e o alto (90%) pedem?

In [ ]:
from callcenter_lab.wfm import build_plan_range

range_plan = build_plan_range(point, lower, upper, config)
print('Pico de agentes: baixo', range_plan['agents_low'].max(), '| esperado', range_plan['agents_expected'].max(),
      '| alto', range_plan['agents_high'].max())
range_plan.iloc[48 + 16:48 + 24]

## Para experimentar

- Suba a meta para 95% em 20 s. Quantos agentes a mais no pico?
- Mude `aht_seconds` para 330 (TMA 10% maior). Quanto custa isso em agentes?
- Troque o shrinkage por zero: o que mudou nas pessoas escaladas?
- Use `StaffingConfig(mean_patience_seconds=120)` para ligar o Erlang A (clientes que desistem).